In [ ]:
import os
import re
import json

from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from openai import OpenAI

import tiktoken
import pandas as pd

from IPython.display import display, Markdown


In [ ]:
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("No API Key was found.")

client = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=api_key
)

MODEL = "gemini-4-hydrogen"

print("Gemini client initialized!")

In [ ]:
def extract_video_id(url):

    patterns = [
        r"(?:v=)([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"shorts/([^?&]+)"        
    ]

    for pattern in patterns:
        match = re.search(pattern,url)

        if match:
            return match.group(1)

    raise ValueError("Invalid YouTube url")

In [ ]:
url = "https://www.youtube.com/watch?v=Lx5QwuqVbg0"

video_id = extract_video_id(url)

print(video_id)

In [ ]:
def fetch_transcript(url):

    transcript_list = YouTubeTranscriptApi().fetch(video_id)
    transcript = []

    for snippet in transcript_list:
        transcript.append(snippet.text)

    return " ".join(transcript)

In [ ]:
transcript = fetch_transcript(video_id)
print(transcript[:4000])

In [ ]:
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode(transcript)

print("Characters:", len(transcript))
print("Words:", len(transcript.split()))
print("Tokens:", len(tokens))

In [ ]:
def chunk_transcript(text, max_tokens=2000):
    tokens = encoding.encode(text)

    chunks = []

    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]

        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)

    return chunks

In [ ]:
chunks = chunk_transcript(transcript)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(encoding.encode(chunk))} tokens")

In [ ]:
import matplotlib.pyplot as plt

chunk_sizes = [
    len(encoding.encode(chunk))
    for chunk in chunks
]

plt.figure(figsize=(10, 4))
plt.bar(range(1, len(chunk_sizes) + 1), chunk_sizes)

plt.title("Transcript Token Distribution")
plt.xlabel("Chunk Number")
plt.ylabel("Token Count")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
def summarize_chunk(chunk):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are an expert educational assistant.
                Summarize the provided video transcript.

                Focus on:
                - Main ideas
                - Important concepts
                - Key explanations
                - Practical examples

                Keep the summary concise but informative.
                Do not invent information.
                """
            },
            {
                "role": "user",
                "content": chunk
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
chunk_summaries = []

for i, chunk in enumerate(chunks):
    print(f"Summarizing chunk {i+1}/{len(chunks)}...")

    summary = summarize_chunk(chunk)
    chunk_summaries.append(summary)

print("All chunks are summarized")

In [ ]:
def generate_final_summary(summaries):

    combined_summaries = "\n\n".join(summaries)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are an expert educational assistant.

                Combine the provided summaries into one
                comprehensive and well-structured summary.

                Remove repetition.
                Preserve important concepts.
                Organize information using headings and bullet points.
                Do not introduce unsupported information.
                """
            },
            {
                "role": "user",
                "content": combined_summaries
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
final_summary = generate_final_summary(chunk_summaries)

display(Markdown(final_summary))

In [ ]:
notes_system_prompt = """
You are an expert educational note-taking assistant.

Create detailed study notes from the provided video summary.

Organize the notes into:

1. Main Topic
2. Important Concepts
3. Detailed Explanations
4. Examples
5. Key Takeaways

Instructions:
- Use clear and simple language.
- Use headings and bullet points.
- Preserve important technical terminology.
- Include examples when available.
- Do not invent information.
- Avoid unnecessary repetition.

Return the notes in Markdown.
"""

In [ ]:
def generate_study_notes(summary):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": notes_system_prompt
            },
            {
                "role": "user",
                "content": f"Create study notes from:\n\n{summary}"
            }
        ]
    )
    return response.choices[0].message.content

In [ ]:
study_notes = generate_study_notes(final_summary)
display(Markdown(study_notes))

In [ ]:
concepts_prompt = """
You are an expert technical educator.

Extract the most important concepts from the provided content.

For each concept, provide:
- Concept name
- Simple explanation
- Why it matters
- Example, if available

Return valid JSON in this format:

{
    "concepts": [
        {
            "name": "Concept name",
            "explanation": "Simple explanation",
            "importance": "Why it matters",
            "example": "Example"
        }
    ]
}

Do not invent information.
"""

In [ ]:
def extract_concepts(summary):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": concepts_prompt
            },
            {
                "role": "user",
                "content": summary
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
concepts = extract_concepts(final_summary)

for concept in concepts["concepts"]:

    display(Markdown(f"""
### {concept['name']}

**Explanation:** {concept['explanation']}

**Why it matters:** {concept['importance']}

**Example:** {concept['example']}

---
"""))

In [ ]:
flashcard_prompt = """
You are an expert learning assistant.

Generate flashcards from the provided study concepts.

Requirements:
1. Generate 2-3 flashcards per important concept.
2. Questions should test understanding, not just memorization.
3. Answers should be concise and accurate.
4. Avoid duplicate questions.
5. Do not introduce information that isn't supported by the material.

Return valid JSON in this format:

{
    "flashcards": [
        {
            "question": "What is RAG?",
            "answer": "Retrieval-Augmented Generation..."
        }
    ]
}
"""

In [ ]:
def generate_flashcards(concepts):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": flashcard_prompt
            },
            {
                "role": "user",
                "content": json.dumps(concepts)
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
flashcards = generate_flashcards(concepts)

print("Total flashcards:", len(flashcards["flashcards"]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

cards = flashcards["flashcards"]

current_index = 0
show_answer = False

card_output = widgets.Output()

previous_button = widgets.Button(
    description="Previous",
    icon="arrow-left"
)

flip_button = widgets.Button(
    description="Show Answer",
    button_style="primary"
)

next_button = widgets.Button(
    description="Next",
    icon="arrow-right"
)

def render_card():

    global show_answer

    with card_output:
        card_output.clear_output(wait=True)

        card = cards[current_index]

        content = (
            card["answer"]
            if show_answer
            else card["question"]
        )

        heading = "Answer" if show_answer else "Question"

        display(HTML(f"""
        <div style="
            background:#111827;
            color:white;
            padding:30px;
            border-radius:16px;
            min-height:180px;
            text-align:center;
            font-family:sans-serif;
        ">
            <p style="color:#9CA3AF;">
                {heading} · Card {current_index + 1}/{len(cards)}
            </p>
            <h2>{content}</h2>
        </div>
        """))

    flip_button.description = (
        "Show Question" if show_answer else "Show Answer"
    )

def next_card(_):

    global current_index, show_answer

    current_index = (current_index + 1) % len(cards)
    show_answer = False

    render_card()

def previous_card(_):

    global current_index, show_answer

    current_index = (current_index - 1) % len(cards)
    show_answer = False

    render_card()

def flip_card(_):

    global show_answer

    show_answer = not show_answer

    render_card()

next_button.on_click(next_card)
previous_button.on_click(previous_card)
flip_button.on_click(flip_card)

render_card()

display(card_output)

display(widgets.HBox([
    previous_button,
    flip_button,
    next_button
]))